# Pipeline tự động tính doanh thu CVKD theo tháng

Pipeline thực hiện toàn bộ quy trình:

1. Kết nối PostgreSQL.
2. Lấy dữ liệu doanh thu từ `union_all_data_cap_don`.
3. Ghép thông tin nhân sự theo nguyên tắc **match SĐT trước → nếu không có mới match Họ tên**.
4. Lọc CVKD theo Channel/Chức danh.
5. Tổng hợp doanh thu theo nhân viên và tháng.
6. Tự động tạo các cột doanh thu theo từng tháng trong khoảng `START_DATE → END_DATE`.
7. Xuất Excel gồm dữ liệu tổng hợp và dữ liệu monthly raw.

> Chỉ cần thay `START_DATE`, `END_DATE` hoặc bật `AUTO_END_DATE`.


In [ ]:
# 1. IMPORT + CẤU HÌNH

import os
from datetime import date, datetime

import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

# =========================
# DATABASE
# =========================
DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Sales Aff"

# Khuyến nghị đặt mật khẩu trong biến môi trường:
# Windows PowerShell:
#   $env:SALES_AFF_DB_PASSWORD="mat_khau"
#
# Nếu chưa có biến môi trường, notebook sẽ yêu cầu nhập mật khẩu.
DB_PASSWORD = os.getenv("SALES_AFF_DB_PASSWORD")

if not DB_PASSWORD:
    from getpass import getpass
    DB_PASSWORD = getpass("Nhập mật khẩu PostgreSQL: ")

# =========================
# KHOẢNG THỜI GIAN
# =========================
START_DATE = "2026-04-01"
END_DATE = "2026-09-30"

# True  -> tự động lấy đến ngày hiện tại
# False -> dùng END_DATE bên trên
AUTO_END_DATE = False

if AUTO_END_DATE:
    END_DATE = date.today().isoformat()

# Chuẩn hóa và kiểm tra ngày
START_DATE = pd.to_datetime(START_DATE).date()
END_DATE = pd.to_datetime(END_DATE).date()

if START_DATE > END_DATE:
    raise ValueError("START_DATE không được lớn hơn END_DATE.")

print(f"Khoảng dữ liệu: {START_DATE} -> {END_DATE}")


In [ ]:
# 2. KẾT NỐI POSTGRESQL

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(
    connection_string,
    pool_pre_ping=True
)

# Kiểm tra kết nối
with engine.connect() as conn:
    test_value = conn.execute(text("SELECT 1")).scalar()

if test_value != 1:
    raise ConnectionError("Không kiểm tra được kết nối PostgreSQL.")

print("Kết nối PostgreSQL thành công.")


In [ ]:
# 3. SQL LẤY DOANH THU DẠNG LONG

SQL = 'WITH order_total AS\n(\n    SELECT\n        uadcd."Code sale",\n\n        COALESCE(dsns_phone."Họ tên", dsns_name."Họ tên") AS "Họ tên",\n        COALESCE(dsns_phone."Điện thoại", dsns_name."Điện thoại") AS "Điện thoại",\n\n        uadcd."Số hợp đồng",\n        uadcd."Số tiền thanh toán",\n\n        COALESCE(\n            dsns_phone."QUẢN LÝ CẤP 1 (BDM)",\n            dsns_name."QUẢN LÝ CẤP 1 (BDM)"\n        ) AS "QUẢN LÝ CẤP 1 (BDM)",\n\n        COALESCE(\n            dsns_phone."QUẢN LÝ CẤP 2 (BDD)",\n            dsns_name."QUẢN LÝ CẤP 2 (BDD)"\n        ) AS "QUẢN LÝ CẤP 2 (BDD)",\n\n        COALESCE(\n            dsns_phone."Quản lý Cấp 3 (BDH)",\n            dsns_name."Quản lý Cấp 3 (BDH)"\n        ) AS "Quản lý Cấp 3 (BDH)",\n\n        COALESCE(\n            dsns_phone."Channal",\n            dsns_name."Channal"\n        ) AS "Channal",\n\n        COALESCE(\n            dsns_phone."Chức danh",\n            dsns_name."Chức danh"\n        ) AS "Chức danh",\n\n        uadcd."Ngày thanh toán"::date AS "Ngày thanh toán",\n\n        EXTRACT(\n            YEAR FROM uadcd."Ngày thanh toán"::date\n        )::int AS "Năm thanh toán",\n\n        EXTRACT(\n            MONTH FROM uadcd."Ngày thanh toán"::date\n        )::int AS "Tháng thanh toán"\n\n    FROM public.union_all_data_cap_don uadcd\n\n    -- Match SĐT trước\n    LEFT JOIN public.ds_nhan_su_affina dsns_phone\n        ON TRIM(LEADING \'0\' FROM TRIM(uadcd."Code sale"))\n         = TRIM(LEADING \'0\' FROM TRIM(dsns_phone."Điện thoại"))\n\n    -- Chỉ fallback sang Họ tên khi KHÔNG match được SĐT\n    LEFT JOIN public.ds_nhan_su_affina dsns_name\n        ON dsns_phone."Điện thoại" IS NULL\n       AND UPPER(TRIM(uadcd."Code sale"))\n         = UPPER(TRIM(dsns_name."Họ tên"))\n\n    WHERE uadcd."Ngày thanh toán"::date >= CAST(:start_date AS DATE)\n      AND uadcd."Ngày thanh toán"::date < CAST(:end_date AS DATE) + INTERVAL \'1 day\'\n),\n\ncvkd AS\n(\n    SELECT\n        UPPER(TRIM("Họ tên")) AS "Họ tên",\n        "Điện thoại",\n\n        STRING_AGG(\n            DISTINCT "Channal",\n            \', \'\n            ORDER BY "Channal"\n        ) AS "Channal",\n\n        STRING_AGG(\n            DISTINCT "Chức danh",\n            \', \'\n            ORDER BY "Chức danh"\n        ) AS "Chức danh",\n\n        "Năm thanh toán",\n        "Tháng thanh toán",\n\n        SUM("Số tiền thanh toán") AS total\n\n    FROM order_total\n\n    WHERE "Channal" IN (\n        \'Core Agency\',\n        \'Neo\',\n        \'CTV_TSA (TSA2)\'\n    )\n\n    AND "Chức danh" IN (\n        \'DSA\',\n        \'AG\',\n        \'TSA 2\'\n    )\n\n    GROUP BY\n        UPPER(TRIM("Họ tên")),\n        "Điện thoại",\n        "Năm thanh toán",\n        "Tháng thanh toán"\n)\n\nSELECT\n    "Họ tên",\n    "Điện thoại",\n    "Channal",\n    "Chức danh",\n    "Năm thanh toán",\n    "Tháng thanh toán",\n    total\nFROM cvkd\nORDER BY\n    "Họ tên",\n    "Năm thanh toán",\n    "Tháng thanh toán";\n'

print("Đã nạp SQL.")

In [ ]:
# 4. CHUẨN BỊ PARAMETER + KIỂM TRA

params = {
    "start_date": START_DATE,
    "end_date": END_DATE
}

print("Parameters:")
print(params)


In [ ]:
# 5. CHẠY SQL

with engine.connect() as conn:
    df_raw = pd.read_sql(
        text(SQL),
        conn,
        params=params
    )

# Chuẩn hóa kiểu dữ liệu
if not df_raw.empty:
    df_raw["total"] = pd.to_numeric(
        df_raw["total"],
        errors="coerce"
    ).fillna(0)

print(f"Đã lấy {len(df_raw):,} dòng.")

display(df_raw.head(10))


In [ ]:
# 6. PIVOT TỰ ĐỘNG THEO THÁNG

if df_raw.empty:
    raise ValueError(
        f"Không có dữ liệu từ {START_DATE} đến {END_DATE}."
    )

# Các cột định danh nhân viên
index_cols = [
    "Họ tên",
    "Điện thoại",
    "Channal",
    "Chức danh"
]

# Tạo danh sách tháng thực tế trong khoảng báo cáo.
# Dùng cả năm + tháng để không bị trùng nếu khoảng báo cáo đi qua nhiều năm.
periods = pd.period_range(
    start=pd.Period(START_DATE, freq="M"),
    end=pd.Period(END_DATE, freq="M"),
    freq="M"
)

# Tạo key YYYY-MM để pivot ổn định
df_raw["month_key"] = (
    pd.to_datetime(
        df_raw["Năm thanh toán"].astype(str)
        + "-"
        + df_raw["Tháng thanh toán"].astype(str).str.zfill(2)
        + "-01"
    ).dt.to_period("M")
)

pivot = (
    df_raw.pivot_table(
        index=index_cols,
        columns="month_key",
        values="total",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Đảm bảo tất cả tháng trong khoảng START_DATE -> END_DATE đều tồn tại,
# kể cả tháng không có doanh thu.
month_cols = []

for period in periods:
    source_col = period
    output_col = f"total_{period.year}_{period.month:02d}"

    month_cols.append(output_col)

    if source_col in pivot.columns:
        pivot = pivot.rename(
            columns={source_col: output_col}
        )
    else:
        pivot[output_col] = 0

# Tổng doanh thu toàn bộ khoảng thời gian
pivot["total"] = pivot[month_cols].sum(axis=1)

# Sắp xếp cột
df_result = pivot[
    index_cols + ["total"] + month_cols
].copy()

# Sắp xếp nhân viên
df_result = df_result.sort_values(
    by=["Họ tên", "Điện thoại"],
    na_position="last"
).reset_index(drop=True)

print("Các cột doanh thu theo tháng:")
print(month_cols)

display(df_result.head(10))


In [ ]:
# 7. KIỂM TRA KẾT QUẢ

print("========== KIỂM TRA ==========")
print(f"Số nhân viên: {len(df_result):,}")
print(f"Số dòng raw monthly: {len(df_raw):,}")
print(f"Số cột kết quả: {len(df_result.columns):,}")

# Kiểm tra tổng doanh thu giữa raw và pivot
raw_total = df_raw["total"].sum()
pivot_total = df_result["total"].sum()

print(f"Tổng doanh thu raw   : {raw_total:,.0f}")
print(f"Tổng doanh thu pivot : {pivot_total:,.0f}")

if abs(raw_total - pivot_total) > 0.01:
    raise ValueError(
        "Tổng doanh thu raw và pivot không khớp. "
        "Cần kiểm tra lại logic pivot."
    )

print("✓ Tổng doanh thu khớp.")


In [ ]:
# 8. XUẤT EXCEL

output_file = (
    f"doanh_thu_cvkd_"
    f"{START_DATE.strftime('%Y%m%d')}_"
    f"{END_DATE.strftime('%Y%m%d')}.xlsx"
)

# Không xuất cột kỹ thuật month_key vào Excel
df_raw_export = df_raw.drop(
    columns=["month_key"],
    errors="ignore"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # Sheet chính: mỗi nhân viên một dòng
    df_result.to_excel(
        writer,
        sheet_name="CVKD",
        index=False
    )

    # Sheet dữ liệu dạng long: mỗi nhân viên - tháng một dòng
    df_raw_export.to_excel(
        writer,
        sheet_name="Raw_Monthly",
        index=False
    )

print(f"Đã xuất Excel: {output_file}")
print(f"Đường dẫn: {os.path.abspath(output_file)}")


## Logic vận hành

- Không cần sửa SQL khi sang tháng mới.
- `START_DATE` và `END_DATE` quyết định khoảng dữ liệu.
- `AUTO_END_DATE = True` sẽ tự lấy đến ngày chạy notebook.
- SQLAlchemy dùng `CAST(:start_date AS DATE)` và `CAST(:end_date AS DATE)` để tránh lỗi với cú pháp PostgreSQL `:end_date::date`.
- Kết quả chính nằm ở sheet `CVKD`.
- Dữ liệu chi tiết theo tháng nằm ở sheet `Raw_Monthly`.
- Cột tháng có dạng `total_2026_04`, `total_2026_05`, ... để tránh nhầm tháng nếu báo cáo đi qua nhiều năm.

### Lưu ý về dữ liệu nhân sự

Logic JOIN hiện tại là:

**SĐT → nếu không match SĐT → match Họ tên.**

Nếu bảng `ds_nhan_su_affina` có nhiều dòng trùng cùng một SĐT hoặc cùng một Họ tên, JOIN vẫn có thể làm nhân bản doanh thu. Khi đó cần bổ sung bước `ROW_NUMBER()` để chọn một bản ghi nhân sự duy nhất trước khi JOIN.
